# Reasoner Notebook — PELLET

Notebook này chạy **Pellet Reasoner** trên file `v-idiomV5.rdf`.

| | Pellet | HermiT |
|---|---|---|
| **Tốc độ** | Nhanh hơn (tối ưu cho dữ liệu thực thể) | Chậm hơn |
| **Độ chính xác** | Tốt với ABox (individual instances) | Mạnh hơn với TBox phức tạp |
| **Yêu cầu** | Java | Java |

> **⚠️ File `v-idiomV5.rdf` gốc sẽ KHÔNG bị thay đổi.**

In [1]:
# CELL 1: TẢI ONTOLOGY
from owlready2 import *
BASE_IRI = "http://www.semanticweb.org/phandangvu/ontologies/2026/8/v-idiomv2#"

temp_world = World()
onto_r = temp_world.get_ontology("../../ontology_protege/v-idiomV5.rdf").load()
print("✅ Đã tải:", onto_r.base_iri)

✅ Đã tải: http://www.semanticweb.org/phandangvu/ontologies/2026/8/v-idiomv5#


In [2]:
# CELL 2: CHẠY PELLET REASONER + ĐO THỜI GIAN
import time

print("🔵 Đang chạy PELLET Reasoner...")
t_start = time.time()

with onto_r:
    sync_reasoner_pellet(x=temp_world)

t_elapsed = time.time() - t_start
print(f"✅ PELLET chạy xong! Thời gian: {t_elapsed:.3f} giây")

🔵 Đang chạy PELLET Reasoner...


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/jena-arq-fixed2.10.0.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/httpclient-4.2.3.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/aterm-java-1.6.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/xercesImpl-2.12.2.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/slf4j-api-1.6.4.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/jena-iri-0.9.5.jar:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/pellet/owlapi-

✅ PELLET chạy xong! Thời gian: 0.725 giây


* Owlready2 * Pellet took 0.7190158367156982 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [3]:
# CELL 3: XÂY DỰNG INDEX
Khung_cls = temp_world[BASE_IRI + "Khung_Bản_Thể_học"]
VN_cls    = temp_world[BASE_IRI + "Vietnamese_idiom"]
EN_cls    = temp_world[BASE_IRI + "English_Idiom"]

frame_index = {}
for khung in Khung_cls.instances():
    frame_index[khung.name] = {"obj": khung, "vn": [], "en": []}

for vi_id in VN_cls.instances():
    for khung in (vi_id.Có_khung or []):
        if khung.name in frame_index:
            frame_index[khung.name]["vn"].append(vi_id.name.replace('_', ' '))

for en_id in EN_cls.instances():
    for khung in (en_id.Có_khung or []):
        if khung.name in frame_index:
            frame_index[khung.name]["en"].append(en_id.name.replace('_', ' '))

print(f"✅ Đã lập chỉ mục {len(frame_index)} Khung.")

✅ Đã lập chỉ mục 84 Khung.


In [4]:
# CELL 4: ĐỒNG NGHĨA ANH - VIỆT
def print_khung(name, info):
    khung = info["obj"]
    print(f"\n📦 {name}")
    if khung.Có_bối_cảnh:  print(f"   Bối cảnh : {khung.Có_bối_cảnh[0].name}")
    if khung.Có_hành_động: print(f"   Hành động: {khung.Có_hành_động[0].name}")
    if khung.Có_kết_quả:   print(f"   Kết quả  : {khung.Có_kết_quả[0].name}")
    if khung.Có_mục_đích:  print(f"   Mục đích : {khung.Có_mục_đích[0].name}")
    for vn in info["vn"]: print(f"   🇻🇳  {vn}")
    for en in info["en"]: print(f"   🇬🇧  {en}")
    print("-" * 65)

print("=" * 65)
print("  [PELLET] CỤM ĐỒNG NGHĨA ANH - VIỆT")
print("=" * 65)
count = 0
for name, info in sorted(frame_index.items()):
    if info["vn"] and info["en"]:
        count += 1
        print_khung(name, info)
print(f"\n✅ Tổng cộng: {count} cụm đồng nghĩa Anh - Việt.")

  [PELLET] CỤM ĐỒNG NGHĨA ANH - VIỆT

📦 Khung10
   Bối cảnh : BC_Xung_đột
   Kết quả  : KQ_Bị_khống_chế
   🇻🇳  Kẻ cắp gặp bà già bằng mới
   🇻🇳  Vỏ quýt dày có móng tay nhọn
   🇬🇧  Diamond cut diamond
-----------------------------------------------------------------

📦 Khung11
   Bối cảnh : BC_Rủi_ro
   Hành động: HD_Liều_lĩnh
   Kết quả  : KQ_Gieo_tai_họa
   🇻🇳  Đi đêm lắm có ngày gặp ma
   🇬🇧  The pitcher goes so often to the well that it is broken at last
-----------------------------------------------------------------

📦 Khung12
   Bối cảnh : BC_Sinh_hoạt
   Hành động: HD_Trì_hoãn
   🇻🇳  Mưa nào mát mặt bấy giờ
   🇬🇧  Live from hand to mouth
-----------------------------------------------------------------

📦 Khung13
   Bối cảnh : BC_Bảo_mật
   Kết quả  : KQ_Lộ_tẩy
   🇻🇳  Cháy nhà ra mặt chuột
   🇬🇧  Truth will out
-----------------------------------------------------------------

📦 Khung14
   Bối cảnh : BC_Đối_nhân_xử_thế
   Hành động: HD_Bắt_chước
   Kết quả  : KQ_Hình_thành_nhâ

In [5]:
# CELL 5: ĐỒNG NGHĨA NỘI BỘ
print("=" * 65)
print("  [PELLET] ĐỒNG NGHĨA NỘI BỘ (≥2 câu cùng ngôn ngữ)")
print("=" * 65)
count = 0
for name, info in sorted(frame_index.items()):
    if len(info["vn"]) >= 2 or len(info["en"]) >= 2:
        count += 1
        print_khung(name, info)
print(f"\n✅ Tổng cộng: {count} cụm đồng nghĩa nội bộ.")

  [PELLET] ĐỒNG NGHĨA NỘI BỘ (≥2 câu cùng ngôn ngữ)

📦 Khung10
   Bối cảnh : BC_Xung_đột
   Kết quả  : KQ_Bị_khống_chế
   🇻🇳  Kẻ cắp gặp bà già bằng mới
   🇻🇳  Vỏ quýt dày có móng tay nhọn
   🇬🇧  Diamond cut diamond
-----------------------------------------------------------------

📦 Khung16
   Bối cảnh : BC_Thương_mại
   Kết quả  : KQ_Tổn_thất_tài_sản
   🇻🇳  Của rẻ là của ôi
   🇻🇳  Của thiên trả địa
   🇬🇧  Cheap things are not good
   🇬🇧  Easy come easy go
-----------------------------------------------------------------

📦 Khung21
   Bối cảnh : BC_Giao_tiếp
   Hành động: HD_Thổi_phồng
   🇻🇳  Múa rìu qua mắt thợ
   🇻🇳  Thùng rỗng kêu to
   🇬🇧  Empty vessels make the most noise
   🇬🇧  Teach fish to swim
-----------------------------------------------------------------

📦 Khung24
   Bối cảnh : BC_Lao_động
   Hành động: HD_Nỗ_lực
   Kết quả  : KQ_Thành_công
   🇻🇳  Có công mài sắt có ngày nên kim
   🇻🇳  Không làm mà đòi có ăn
   🇬🇧  No pain no gain
   🇬🇧  Practice makes perfect
----------